## **Argument Scoring Extraction and Analysis**
*This Colab notebook retrieves and analyzes the score values assigned by the model to each argument, enabling a structured assessment of argumentative strength.*

**Models**:
* Gemma: 3 - 4b
* Llama: 3.2 - 3b
* Deepseek: V2 - 16b

**Prompt**: 1 shot


In [41]:
import pandas as pd
import json
import re

## Loading Data 
Scoring models: 
* Llama vs Gemma
* Deepseek vs Llama
* Gemma vs Deepseek

Master file containing all the arguments generated:
* master_args.csv

In [42]:
master_args = pd.read_csv('../master_args.csv')

#### Gemma


In [9]:
JGemma_ds_ll = pd.read_csv('gemma/20250814-174849gemmaargument_dsargument_llama_v1.csv')
JGemma_ge_ds = pd.read_csv('gemma/20250815-165106gemmaargument_gemmaargument_ds_v1.csv')
JGemma_ll_ge = pd.read_csv('gemma/20250813-160613gemma_1shot_all_v1.csv') #llama vs gemma


In [11]:
JGemma_ds_ll.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2971 entries, 0 to 2970
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   topic      2971 non-null   int64  
 1   response   2971 non-null   object 
 2   prompt     2971 non-null   object 
 3   timestamp  2971 non-null   object 
 4   op1        2971 non-null   object 
 5   op2        2971 non-null   object 
 6   op3        2971 non-null   object 
 7   duration   2971 non-null   float64
dtypes: float64(1), int64(1), object(6)
memory usage: 185.8+ KB


In [25]:
JGemma_ds_ll = extract_scores(JGemma_ds_ll)
JGemma_ge_ds = extract_scores(JGemma_ge_ds)
JGemma_ll_ge = extract_scores(JGemma_ll_ge)

Error decoding 'comments' array from string: Extra data: line 1 column 282 (char 281)
Problematic comments string: '["This argument is reasonably well-structured and aligns with the opinions by acknowledging the cour...'


In [39]:
JGemma_ds_ll[['score1', 'score2']] = JGemma_ds_ll['scores'].apply(parse_scores_ast)
JGemma_ge_ds[['score1', 'score2']] = JGemma_ge_ds['scores'].apply(parse_scores_ast)
JGemma_ll_ge[['score1', 'score2']] = JGemma_ll_ge['scores'].apply(parse_scores_ast)
JGemma_ds_ll['arg1']=master_args['argument_ds']
JGemma_ge_ds['arg1']=master_args['argument_gemma']
JGemma_ll_ge['arg1']=master_args['argument_llama']
JGemma_ds_ll['arg2']=master_args['argument_llama']
JGemma_ge_ds['arg2']=master_args['argument_ds']
JGemma_ll_ge['arg2']=master_args['argument_gemma']
JGemma_ds_ll['color']=master_args['color']
JGemma_ge_ds['color']=master_args['color']
JGemma_ll_ge['color']=master_args['color']
JGemma_ds_ll['stance']=master_args['stance']
JGemma_ge_ds['stance']=master_args['stance']
JGemma_ll_ge['stance']=master_args['stance']

In [36]:
JGemma_ds_ll.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2971 entries, 0 to 2970
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   topic      2971 non-null   int64  
 1   response   2971 non-null   object 
 2   prompt     2971 non-null   object 
 3   timestamp  2971 non-null   object 
 4   op1        2971 non-null   object 
 5   op2        2971 non-null   object 
 6   op3        2971 non-null   object 
 7   duration   2971 non-null   float64
 8   scores     2971 non-null   object 
 9   comments   2947 non-null   object 
 10  score1     2971 non-null   int64  
 11  score2     2971 non-null   int64  
 12  arg1       2971 non-null   object 
 13  arg2       2971 non-null   object 
dtypes: float64(1), int64(3), object(10)
memory usage: 325.1+ KB


In [18]:
JGemma_ds_ll.head(10)

topic  \
0      0   
1      1   
2      2   
3      3   
4      4   
5      5   
6      6   
7      7   
8      8   
9      9   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                response  \
0                        ```json\n{\n  "scores": [4, 3],\n  "comments": ["This argument is well-structured and logically connects the opinions by highlighting Jay Inslee and Beto O'Rourke as leaders and advocating for increased support for their efforts. It effectively uses the provided opinions to build a cohesive argument.",\n  "This argument is somewhat structured but lacks a strong connection to the specific opinions. While it acknowledges the importance of climate change and the need for leadership, it doesn't explicitly reference the provided opinions about Inslee's commitment or the movement surrounding him."]\n}\n```   
1                                                                                   ```json\n{\n  "scores": [4, 5],\n  "comments": ["This argument is well-structured and logically connects the opinions, clearly outlining the urgency of the situation, the actions of individuals and groups, and the call for government action. It effectively synthesizes the provided viewpoints.",\n  "This argument is excellent: clear, logical, and fully aligned with the opinions. It presents a strong case for immediate climate action, referencing the urgency, individual and collective efforts, and the need for government intervention."]\n}\n```   
2                                                           ```json\n{\n  "scores": [4, 3],\n  "comments": ["This argument is well-structured and logically connects the opinions, clearly articulating the perception of unpredictable weather in Chicago and linking it to the concept of global warming. It effectively summarizes the provided viewpoints.", "This argument is somewhat structured but lacks depth. While it acknowledges the opinions, it doesn't fully elaborate on the connection between the specific events (April snow, May heat) and the broader issue of global warming. It's a basic restatement of the opinions."]\n}\n```   
3  ```json\n{\n  "scores": [4, 3],\n  "comments": ["This argument is well-structured and effectively connects the provided opinions, clearly outlining the importance of youth-led movements like Earth Strike and the broader call for action. The flow of ideas is logical and the argument is aligned with the opinions.",\n  "This argument is somewhat structured but lacks a strong connection to the provided opinions. While it acknowledges the urgency of climate change and mentions relevant movements, it doesn't explicitly integrate the specific details from the opinions, such as the Earth Strike or Jay Inslee's movement."]\n}\n```   
4                                                                                                        ```json\n{\n  "scores": [4, 2],\n  "comments": ["This argument is well-structured and aligns with the opinions, focusing on the vulnerability of the young person and the potential manipulation. It logically connects the concerns about climate change and the need to support those raising awareness.", "This argument is weak and doesn't directly address the opinions. It's overly focused on accusations of manipulation and doesn't clearly link to the concerns about future generations and the impact of climate change."]\n}\n```   
5                                  ```json\n{\n  "scor

In [40]:
JGemma_ds_ll.to_csv('gemma/jgemma_ds_ll.csv',index=False)
JGemma_ge_ds.to_csv('gemma/jgemma_ge_ds.csv',index=False)
JGemma_ll_ge.to_csv('gemma/jgemma_ll_ge.csv',index=False)

In [26]:
JGemma_ds_ll.head(1)

,topic,response,prompt,timestamp,op1,op2,op3,duration,scores,comments
0,0,"```json\n{\n ""scores"": [4, 3],\n ""comments"": [""This argument is well-structured and logically connects the opinions by highlighting Jay Inslee and Beto O'Rourke as leaders and advocating for increased support for their efforts. It effectively uses the provided opinions to build a cohesive argument."",\n ""This argument is somewhat structured but lacks a strong connection to the specific opinions. While it acknowledges the importance of climate change and the need for leadership, it doesn't explicitly reference the provided opinions about Inslee's commitment or the movement surrounding him.""]\n}\n```","The goal of this task is to evaluate the quality of the 2 arguments generated based on a set of opinions about a specific climate change topic. Check the structure and effectiveness of the arguments, not their factual validity. The arguments should logically align with the provided opinions and demonstrate clear reasoning. \n\nAssign a score from 1 (worst) to 5 (best) to each argument based on the following criteria:\nScore: 1 - The argument is poorly structured, incoherent, or unrelated to the opinions.\nScore: 2 - The argument is weakly connected to the opinions and lacks clarity or logic.\nScore: 3 - The argument is somewhat structured but has minor flaws in logic or relevance.\nScore: 4 - The argument is well-structured, logical, and mostly aligned with the opinions.\nScore: 5 - The argument is excellent: clear, logical, and fully aligned with the opinions.\n\nReturn your output in JSON format using the following structure:\nOutput = {\n""Score"": ""[ score_for_argument_1, score_for_argument_2]"",\n""comments"":""[comment_for_argument_1, comment_for_argument_2]""\n}\n\nThis is an example\n**Example Input:**\n{\n ""opinions"": [""Rising global temperatures are causing more frequent and severe wildfires."", ""Deforestation contributes to climate change by reducing the number of trees that absorb CO2."", ""Governments should invest in reforestation programs to combat climate change.""],\n ""arguments"": [""It is widely acknowledged that rising global temperatures are causing wildfires to become more frequent and severe, with the number of trees that absorb CO2 being significantly reduced due to deforestation. As a result, climate change is being exacerbated, and the restoration of carbon-absorbing trees through reforestation programs is seen as a necessary solution. It is urged that investments in such programs be made by governments to help mitigate the ongoing effects of climate change."",""Climate change is bad because it makes the weather hotter and causes problems for animals.""]\n }\n**Output for the Example Input:**\n{\n ""scores"":[5,2],\n ""comments:"" [""This argument is logically structured and connects the opinions effectively. It clearly links deforestation, wildfires, and the need for reforestation programs."",\n ""This argument is vague and does not clearly connect to the provided opinions. It lacks specificity and logical structure""]\n }\n------------------------------------------\nPlease punctuate the following arguments:\n{\n\t""opinions"":[""""@mediaknowitall Jay Inslee is the candidate committed to combatting climate change. This is our moment! Join the movement: https://t.co/Dw7y86VVWN https://t.co/ZWh4frZnhE "",\n""@AustinJenkinsN3 I'm sure all the other candidates will look at this attack and then think what a great idea it would be to have all the candidates attacked by Inslee in a 'climate change' debate "",\n""Glad to see @BetoORourke join @JayInslee as real leaders on #climatechange — looking forward to seeing this become the #1 priority among the rest of the candidates. https://t.co/p43pIvIzyg ""], ""arguments"": [""It is widely recognized that there are numerous individuals committed to combating climate change, with Jay Inslee and Beto O'Rourke being among them. These real leaders have been advocating for the issue to 

#### Llama


In [45]:
JLlama_ds_ll = pd.read_csv('llama/20250819-020559llamaargument_dsargument_llama_v1.csv')
JLlama_ge_ds = pd.read_csv('llama/20250820-190539llamaargument_gemmaargument_ds_v1.csv')
JLlama_ll_ge = pd.read_csv('llama/20250817-171859llamaargument_llamaargument_gemma_v1.csv')


In [55]:
JLlama_ds_ll['arg1']=master_args['argument_ds']
JLlama_ge_ds['arg1']=master_args['argument_gemma']
JLlama_ll_ge['arg1']=master_args['argument_llama']
JLlama_ds_ll['arg2']=master_args['argument_llama']
JLlama_ge_ds['arg2']=master_args['argument_ds']
JLlama_ll_ge['arg2']=master_args['argument_gemma']
JLlama_ds_ll['color']=master_args['color']
JLlama_ge_ds['color']=master_args['color']
JLlama_ll_ge['color']=master_args['color']
JLlama_ds_ll['stance']=master_args['stance']
JLlama_ge_ds['stance']=master_args['stance']
JLlama_ll_ge['stance']=master_args['stance']

In [ ]:
JLlama_ds_ll = extract_scores(JLlama_ds_ll)
JLlama_ge_ds = extract_scores(JLlama_ge_ds)
JLlama_ll_ge = extract_scores(JLlama_ll_ge)

JLlama_ds_ll[['score1', 'score2']] = JLlama_ds_ll['scores'].apply(parse_scores_ast)
JLlama_ge_ds[['score1', 'score2']] = JLlama_ge_ds['scores'].apply(parse_scores_ast)
JLlama_ll_ge[['score1', 'score2']] = JLlama_ll_ge['scores'].apply(parse_scores_ast)

In [57]:
JLlama_ds_ll.isna().any(axis=1).sum()

np.int64(1592)

In [58]:
# Create a boolean mask where True indicates a "None" value in any column of that row
mask = JLlama_ds_ll.isna().any(axis=1)

# Filter the DataFrame using the mask
JLlama_ds_ll[mask]

topic  \
1         1   
6         6   
7         7   
8         8   
12       12   
...     ...   
2956   2956   
2960   2960   
2961   2961   
2962   2962   
2963   2963   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [64]:
def extract_scores_separated(df):
    """
    Extract Score 1 and Score 2 from the 'response' column and create new columns.
    Only processes rows where score1 is equal to 0.
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing a 'response' column
    
    Returns:
    pandas.DataFrame: DataFrame with added 'score1' and 'score2' columns
    """
    
    def parse_response(response_text):
        """Parse a single response text to extract scores."""
        if pd.isna(response_text):
            return None, None
        
        # Pattern to match "Argument X: ... Score: number"
        pattern = r'Argument\s+(\d+):[^S]*Score:\s*(\d+(?:\.\d+)?)'
        
        matches = re.findall(pattern, response_text, re.IGNORECASE | re.DOTALL)
        
        score1, score2 = None, None
        
        for arg_num, score in matches:
            if arg_num == '1':
                score1 = float(score)
            elif arg_num == '2':
                score2 = float(score)
        
        return score1, score2
    
    # Create a mask for rows that need processing (where score1 is 0)
    mask = df['score1'] == 0
    
    if mask.any():  # Only process if there are rows that need it
        # Apply the parsing function only to rows that need processing
        scores_to_update = df.loc[mask, 'response'].apply(
            lambda x: pd.Series(parse_response(x))
        )
        
        # Update only the rows that need processing
        df.loc[mask, 'score1'] = scores_to_update.iloc[:, 0]
        df.loc[mask, 'score2'] = scores_to_update.iloc[:, 1]
    
    return df

# Alternative approach using a more specific extraction method
def extract_scores_alternative_separated(df):
    """
    Alternative method with more explicit pattern matching.
    Only processes rows where score1 is equal to 0.
    """
    
    # Check if score1 and score2 columns exist, if not create them with 0
    if 'score1' not in df.columns:
        df['score1'] = 0
    if 'score2' not in df.columns:
        df['score2'] = 0
    
    def extract_argument_scores(text):
        if pd.isna(text):
            return None, None
            
        score1, score2 = None, None
        
        # Split by lines and look for score patterns
        lines = text.split('\n')
        current_argument = None
        
        for line in lines:
            line = line.strip()
            
            # Check if this line indicates an argument
            if line.startswith('Argument 1:'):
                current_argument = 1
            elif line.startswith('Argument 2:'):
                current_argument = 2
            
            # Check if this line contains a score
            score_match = re.search(r'Score:\s*(\d+(?:\.\d+)?)', line, re.IGNORECASE)
            if score_match:
                score_value = float(score_match.group(1))
                if current_argument == 1:
                    score1 = score_value
                elif current_argument == 2:
                    score2 = score_value
        
        return score1, score2
    
    # Create a mask for rows that need processing (where score1 is 0)
    mask = df['score1'] == 0
    
    if mask.any():  # Only process if there are rows that need it
        # Apply the extraction only to rows that need processing
        scores_to_update = df.loc[mask, 'response'].apply(
            lambda x: pd.Series(extract_argument_scores(x))
        )
        
        # Update only the rows that need processing
        df.loc[mask, 'score1'] = scores_to_update.iloc[:, 0]
        df.loc[mask, 'score2'] = scores_to_update.iloc[:, 1]
    
    return df



In [67]:
JLlama_ds_ll_processed = extract_scores_alternative_separated(JLlama_ds_ll.copy())

In [69]:
JLlama_ge_ds_processed = extract_scores_alternative_separated(JLlama_ge_ds.copy())
JLlama_ll_ge_processed = extract_scores_alternative_separated(JLlama_ll_ge.copy())

In [71]:
JLlama_ll_ge_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2971 entries, 0 to 2970
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   topic      2971 non-null   int64  
 1   response   2971 non-null   object 
 2   prompt     2971 non-null   object 
 3   timestamp  2971 non-null   object 
 4   op1        2971 non-null   object 
 5   op2        2971 non-null   object 
 6   op3        2971 non-null   object 
 7   duration   2971 non-null   float64
 8   scores     1461 non-null   object 
 9   comments   1313 non-null   object 
 10  arg1       2971 non-null   object 
 11  arg2       2971 non-null   object 
 12  color      2971 non-null   float64
 13  stance     2971 non-null   object 
 14  score1     2349 non-null   float64
 15  score2     2349 non-null   float64
dtypes: float64(4), int64(1), object(11)
memory usage: 371.5+ KB


In [72]:
JLlama_ds_ll_processed.to_csv('llama/jllama_ds_ll.csv',index=False)
JLlama_ge_ds_processed.to_csv('llama/jllama_ge_ds.csv',index=False)
JLlama_ll_ge_processed.to_csv('llama/jllama_ll_ge.csv',index=False)

#### Deepseek

In [73]:
JDeepseek_ds_ll = pd.read_csv('ds/20250819-070659deepseekargument_dsargument_llama_v1.csv')
JDeepseek_ge_ds = pd.read_csv('ds/20250820-012621deepseekargument_gemmaargument_ds_v1.csv')
JDeepseek_ll_ge = pd.read_csv('ds/20250818-152639deepseekargument_llamaargument_gemma_v1.csv') #llama vs gemma

In [74]:
JDeepseek_ds_ll['arg1']=master_args['argument_ds']
JDeepseek_ge_ds['arg1']=master_args['argument_gemma']
JDeepseek_ll_ge['arg1']=master_args['argument_llama']
JDeepseek_ds_ll['arg2']=master_args['argument_llama']
JDeepseek_ge_ds['arg2']=master_args['argument_ds']
JDeepseek_ll_ge['arg2']=master_args['argument_gemma']
JDeepseek_ds_ll['color']=master_args['color']
JDeepseek_ge_ds['color']=master_args['color']
JDeepseek_ll_ge['color']=master_args['color']
JDeepseek_ds_ll['stance']=master_args['stance']
JDeepseek_ge_ds['stance']=master_args['stance']
JDeepseek_ll_ge['stance']=master_args['stance']

In [75]:
JDeepseek_ds_ll = extract_scores(JDeepseek_ds_ll)
JDeepseek_ge_ds = extract_scores(JDeepseek_ge_ds)
JDeepseek_ll_ge = extract_scores(JDeepseek_ll_ge)

JDeepseek_ds_ll[['score1', 'score2']] = JDeepseek_ds_ll['scores'].apply(parse_scores_ast)
JDeepseek_ge_ds[['score1', 'score2']] = JDeepseek_ge_ds['scores'].apply(parse_scores_ast)
JDeepseek_ll_ge[['score1', 'score2']] = JDeepseek_ll_ge['scores'].apply(parse_scores_ast)

Error decoding 'scores' array from string: Expecting value: line 1 column 5 (char 4)
Problematic scores string: '[4, N/A]'
Error decoding 'scores' array from string: Expecting value: line 1 column 5 (char 4)
Problematic scores string: '[4, N/A]'
Error decoding 'comments' array from string: Extra data: line 2 column 1 (char 265)
Problematic comments string: '["This argument is well-structured and logically aligns with the opinions about rising global temper...'
Error decoding 'comments' array from string: Extra data: line 1 column 349 (char 348)
Problematic comments string: '["This argument is well-structured and effectively aligns with the opinions by emphasizing the vulne...'
Error decoding 'comments' array from string: Expecting value: line 1 column 478 (char 477)
Problematic comments string: '["This argument is well-structured and effectively aligns with the opinions by highlighting the impo...'
Error decoding 'comments' array from string: Extra data: line 2 column 1 (char 362)
Prob

In [77]:
JDeepseek_ds_ll.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2971 entries, 0 to 2970
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   topic      2971 non-null   int64  
 1   response   2971 non-null   object 
 2   prompt     2971 non-null   object 
 3   timestamp  2971 non-null   object 
 4   op1        2971 non-null   object 
 5   op2        2971 non-null   object 
 6   op3        2971 non-null   object 
 7   duration   2971 non-null   float64
 8   arg1       2971 non-null   object 
 9   arg2       2971 non-null   object 
 10  color      2971 non-null   float64
 11  stance     2971 non-null   object 
 12  scores     2909 non-null   object 
 13  comments   1811 non-null   object 
 14  score1     2971 non-null   float64
 15  score2     2949 non-null   object 
dtypes: float64(3), int64(1), object(12)
memory usage: 371.5+ KB


In [79]:
JDeepseek_ge_ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2971 entries, 0 to 2970
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   topic      2971 non-null   int64  
 1   response   2971 non-null   object 
 2   prompt     2971 non-null   object 
 3   timestamp  2971 non-null   object 
 4   op1        2971 non-null   object 
 5   op2        2971 non-null   object 
 6   op3        2971 non-null   object 
 7   duration   2971 non-null   float64
 8   arg1       2971 non-null   object 
 9   arg2       2971 non-null   object 
 10  color      2971 non-null   float64
 11  stance     2971 non-null   object 
 12  scores     2936 non-null   object 
 13  comments   1934 non-null   object 
 14  score1     2971 non-null   float64
 15  score2     2969 non-null   float64
dtypes: float64(4), int64(1), object(11)
memory usage: 371.5+ KB


In [80]:
JDeepseek_ll_ge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2971 entries, 0 to 2970
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   topic      2971 non-null   int64  
 1   response   2971 non-null   object 
 2   prompt     2971 non-null   object 
 3   timestamp  2971 non-null   object 
 4   op1        2971 non-null   object 
 5   op2        2971 non-null   object 
 6   op3        2971 non-null   object 
 7   duration   2971 non-null   float64
 8   arg1       2971 non-null   object 
 9   arg2       2971 non-null   object 
 10  color      2971 non-null   float64
 11  stance     2971 non-null   object 
 12  scores     2936 non-null   object 
 13  comments   1697 non-null   object 
 14  score1     2971 non-null   int64  
 15  score2     2971 non-null   int64  
dtypes: float64(2), int64(3), object(11)
memory usage: 371.5+ KB


In [81]:
JDeepseek_ds_ll.to_csv('ds/jds_ds_ll.csv',index=False)
JDeepseek_ge_ds.to_csv('ds/jds_ge_ds.csv',index=False)
JDeepseek_ll_ge.to_csv('ds/jds_ll_ge.csv',index=False)

## Functions to extract information from the LLM response


In [46]:
def process_json_string4(json_str: str) -> tuple:
    """
    Processes a string to extract the 'scores' field using string manipulation/regex,
    rather than relying on full JSON parsing of the entire input.
    Handles strings that might be wrapped in code blocks or have escape characters.

    Args:
        json_str (str): Input string to process.

    Returns:
        tuple: A tuple containing (scores, None).
               Returns (None, None) if 'scores' field is not found or cannot be parsed.
    """
    if not isinstance(json_str, str):
        print("Error: Input must be a string.")
        return None, None

    # Step 1: Clean the input string
    # Remove common code block markers (e.g., ```json, ```)
    cleaned_str = re.sub(r'```json\s*', '', json_str, flags=re.IGNORECASE)
    cleaned_str = re.sub(r'```\s*$', '', cleaned_str).strip()

    # Attempt to handle escaped quotes if they are double-escaped (e.g., \\")
    cleaned_str = cleaned_str.replace('\\\\"', '\\"')

    scores = None
    comments = None # Still keeping for tuple consistency, but will remain None

    # Step 2: Use regex to find the 'scores' field and its value
    # This regex looks for "scores": followed by a JSON array [...]
    # It tries to capture the content of the array.
    # re.DOTALL allows '.' to match newlines, important for multi-line JSON
    match = re.search(r'"scores"\s*:\s*(\[.*?\])', cleaned_str, re.DOTALL)

    if match:
        scores_str = match.group(1) # Get the captured string for the scores array
        try:
            # Step 3: Parse the extracted scores string as JSON
            scores = json.loads(scores_str)
        except json.JSONDecodeError as e:
            print(f"Error decoding 'scores' array from string: {e}")
            print(f"Problematic scores string: '{scores_str}'")
        except Exception as e:
            print(f"An unexpected error occurred while parsing scores: {e}")
    else:
        print("Warning: 'scores' field not found using regex.")

    # Step 3: Use regex to find the 'comments' field and its value
    # This regex looks for "comments": followed by a JSON array [...]
    # We need to be more careful with comments as they can contain nested quotes and brackets
    comments_match = re.search(r'"comments"\s*:\s*(\[.*\])', cleaned_str, re.DOTALL)
    if comments_match:
        comments_str = comments_match.group(1)
        try:
            comments = json.loads(comments_str)
        except json.JSONDecodeError as e:
            print(f"Error decoding 'comments' array from string: {e}")
            print(f"Problematic comments string: '{comments_str[:100]}...'")  # Truncate for readability
        except Exception as e:
            print(f"An unexpected error occurred while parsing comments: {e}")
    else:
        print("Warning: 'comments' field not found using regex.")
        
    return scores, comments

In [47]:
def process_json(row=None):
    return process_json_string4(row.response)

In [48]:
def extract_scores(df): 
    res = df.apply(process_json, axis=1)
    df['scores'], df['comments'] = zip(*res)
    return df


In [49]:
import ast
# Method 1: Using ast.literal_eval (recommended for safety)
def parse_scores_ast(row):
    try:
        if (isinstance(row, str)):
            # Handle None or empty strings
            if pd.isna(row) or row == '' or row is None:
                return pd.Series({'score1': 0, 'score2': 0})
            
            # Remove all spaces from the string before parsing
            cleaned_row = str(row).replace(' ', '')
            
            scores = ast.literal_eval(cleaned_row)
        else: scores = row
              
        # Check if it's a list with at least 2 elements
        if isinstance(scores, list) and len(scores) >= 2:
            return pd.Series({'score1': scores[0], 'score2': scores[1]})
        else:
            # Not a proper list or insufficient elements
            return pd.Series({'score1': 0, 'score2': 0})
            
    except (ValueError, SyntaxError, TypeError) as e:
        # Any parsing error - set to 0
        print(f"Error parsing '{row}': {e}")
        return pd.Series({'score1': 0, 'score2': 0})
